# Vision A threshold baseline (Phase 2, EXP-1)

Scores the clinical threshold baseline on the 129 development cases: core is rCBF < 30% of the healthier hemisphere, and penumbra is Tmax > 6 s, with Tmax below 0 counted as no data. It reports Dice against the final infarct, volume error against `native_volumes.csv`, and how often the predicted core crosses a decision cut-off (21, 31, 51 or 70 mL) when the reference does not. Two variants are scored: core restricted to Tmax > 6 s (the default), and not restricted.

It also measures how far each occlusion mask sits from its nearest Circle of Willis label, which sets the distance limit for the occlusion-site rule in `volumes.py`.

Held-out cases are never loaded. Outputs hold development case IDs, counts and metrics. The logic is in `src/strokecopilot/vision/perfusion/`.

**Before you start**
- Use a CPU runtime: Runtime → Change runtime type → None.

**Run it:** Runtime → Run all, and allow Drive access when asked. Allow about 10 minutes. That is an estimate: the checksum and copy took about 2 minutes in Phase 1, and the scoring time is unmeasured. Section 5 prints progress.

**If it stops:** reconnect and use Runtime → Run all. Nothing is saved until section 7, so a rerun starts clean.

**Send back:** the `OK:` line from section 7, and any error in full. Results are saved to `runs/EXP-1_thresholds_baseline/`, where Claude can read them.

## 1. Settings

In [ ]:
ROOT = '/content/drive/MyDrive/StrokeCopilot'
SLIM_TAR = f'{ROOT}/slim/isles24_slim_v1.tar'
SLIM_SHA256 = 'b0afd16e4cbfb16a69fb5190083fc3869f5dcc6c92f86f10135b0de1f59f4728'
FOLDS_SHA256 = '023813f2dba930b7521653f1549b1fa1369b8bd29b63658175c5cea58410d291'
EXPERIMENT = 'EXP-1'
RUN_DIR = f'{ROOT}/runs/EXP-1_thresholds_baseline'
LOCAL_DIR = '/content/isles24'
EXPECTED_DEVELOPMENT = 129
REPO_REF = 'main'
VARIANTS = {'within_tmax6': True, 'unrestricted': False}  # core restricted to Tmax > 6 s, or not
LIMITS_MM = (3.0, 5.0, 7.5, 10.0, 15.0)
FIGURE_CASES = 6

## 2. Install the package and mount Drive
The branch is pinned to one commit, which every output records. The split's checksum must match; only group sizes are printed.

In [ ]:
import hashlib, json, os, subprocess, sys, time
from google.colab import drive

repo = 'https://github.com/parsa-hke/strokecopilot.git'
found = subprocess.run(['git', 'ls-remote', repo, REPO_REF], capture_output=True, text=True)
if found.returncode or not found.stdout.strip():
    raise RuntimeError(('Could not read the repo. ' + found.stderr)[-800:])
COMMIT = found.stdout.split()[0]
result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'strokecopilot[isles] @ git+{repo}@{COMMIT}'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError(result.stderr[-800:])
del repo, found, result
print(f'Package installed at commit {COMMIT[:7]}.')

drive.mount('/content/drive')

from strokecopilot.data import archive_listing as al
from strokecopilot.data import isles24, splits
from strokecopilot.data.isles24_survey import PENUMBRA_TMAX_S, TISSUE_HU
from strokecopilot.vision.perfusion import evaluate, thresholds, volumes

if hashlib.sha256(open(splits.FOLDS_PATH, 'rb').read()).hexdigest() != FOLDS_SHA256:
    raise ValueError('The installed split is not the fixed one. Stop and check it.')
folds = splits.load_folds()
print(f'Split: {len(folds["heldout"])} held out; folds of ' + ', '.join(str(len(v)) for v in folds['folds'].values()) + '.')
if os.path.exists(os.path.join(RUN_DIR, 'metrics.json')):
    raise FileExistsError(f'{RUN_DIR} already holds a finished run. Nothing was changed. Ask before replacing it.')
os.makedirs(LOCAL_DIR, exist_ok=True)

## 3. Config
Written before the run, as the experiment protocol asks (plan §11).

In [ ]:
config = {
    'experiment': EXPERIMENT,
    'notebook': 'notebooks/11_visionA_thresholds.ipynb',
    'git_commit': COMMIT,
    'data_version': 'isles24_slim_v1.tar',
    'data_sha256': SLIM_SHA256,
    'folds_sha256': FOLDS_SHA256,
    'cases': 'development only, scored per fold 0-4',
    'core_rule': f'rCBF < {thresholds.RCBF_CORE} of the median CBF of the healthier hemisphere (reliable voxels, Tmax <= {thresholds.REFERENCE_MAX_TMAX_S} s)',
    'penumbra_rule': f'Tmax > {PENUMBRA_TMAX_S} s; Tmax < 0 is no data',
    'midline': f'left-right centre of NCCT soft tissue {TISSUE_HU} HU',
    'variants': VARIANTS,
    'reference_volumes': 'native_volumes.csv (infarct_ml, tmax6_ml)',
    'decision_cutoffs_ml': evaluate.DECISION_CUTOFFS_ML,
    'occlusion': {'label_site': {str(k): v for k, v in volumes.LABEL_SITE.items()}, 'provisional_limit_mm': volumes.DISTANCE_LIMIT_MM,
                  'min_site_share': volumes.MIN_SITE_SHARE, 'limits_compared_mm': LIMITS_MM},
}
print(json.dumps(config, indent=1))

## 4. Data
Checks the tarball on Drive and unpacks it to local disk (DEC-3).

In [ ]:
started = time.time()
if al.sha256_of(SLIM_TAR) != SLIM_SHA256:
    raise ValueError('The tarball on Drive is not the one the build recorded. Stop and check it.')
slim_root = isles24.fetch_slim(SLIM_TAR, parent=LOCAL_DIR)
fold_of = {case: int(k) for k, ids in folds['folds'].items() for case in ids}
development = isles24.list_cases(slim_root)
if len(development) != EXPECTED_DEVELOPMENT or set(development) != set(fold_of):
    raise ValueError(f'The loader lists {len(development)} cases, not the {EXPECTED_DEVELOPMENT} development cases.')
print(f'Checked and unpacked in {time.time() - started:.0f} s; {len(development)} development cases.')

## 5. Score every development case
Prints progress every 20 cases.

In [ ]:
NAMES = ('ncct', 'cbf', 'tmax', 'occlusion', 'cow', 'infarct')
rows = {name: [] for name in VARIANTS}
started = time.time()
for i, case in enumerate(development, 1):
    loaded = isles24.load_case(case, slim_root, names=NAMES)
    for name, within in VARIANTS.items():
        rows[name].append({'variant': name, **thresholds.baseline_row(loaded, fold_of[case], core_within_penumbra=within)})
    if i % 20 == 0 or i == len(development):
        print(f'{i}/{len(development)} cases, {time.time() - started:.0f} s', flush=True)
seconds_per_case = (time.time() - started) / len(development)

## 6. Results

In [ ]:
import numpy as np

def pct(values):
    values = [v for v in values if v is not None]
    return [round(float(np.percentile(values, p)), 3) for p in (5, 50, 95)] if values else None

first = rows['within_tmax6']
metrics = {
    'experiment': EXPERIMENT, 'git_commit': COMMIT, 'data_version': 'isles24_slim_v1.tar', 'fold': 'all development, per fold below',
    'cases': len(development), 'seconds_per_case': round(seconds_per_case, 2),
    'variants': {name: evaluate.summarise(rows[name]) for name in VARIANTS},
    'core_not_computed': sum(not r['core_computed'] for r in first),
    'penumbra_error_ml_p5_p50_p95': pct([r['penumbra_error_ml'] for r in first]),
    'penumbra_abs_error_ml_p5_p50_p95': pct([abs(r['penumbra_error_ml']) for r in first]),
    'infarct_reliable_share_p5_p50_p95': pct([r['infarct_reliable_share'] for r in first]),
    'affected_side': {side: sum(r['affected_side'] == side for r in first) for side in ('left', 'right')},
    'occlusion': volumes.limit_table([{k[len('occlusion_'):]: v for k, v in r.items() if k.startswith('occlusion_')} for r in first], LIMITS_MM),
}
for name in VARIANTS:
    s = metrics['variants'][name]['mean_std_over_folds']
    print(f"{name}: Dice {s['dice']['mean']:.3f} ± {s['dice']['std']:.3f}, Dice on reliable Tmax {s['dice_reliable']['mean']:.3f} ± {s['dice_reliable']['std']:.3f}, "
          f"abs volume error {s['abs_error_ml']['mean']:.1f} ± {s['abs_error_ml']['std']:.1f} mL over folds")
    print('  crossings:', metrics['variants'][name]['threshold_crossings'])
print(json.dumps({k: metrics[k] for k in ('core_not_computed', 'penumbra_error_ml_p5_p50_p95', 'infarct_reliable_share_p5_p50_p95', 'affected_side')}, indent=1))
print(json.dumps(metrics['occlusion'], indent=1))

**Figures.** `core_montage.png`: cases spread across infarct volume, with infarct in red and threshold core in cyan. `occlusion_montage.png`: cases nearest a basilar or ICA label, to check the distance limit by eye. Development cases only.

In [ ]:
by_volume = sorted(first, key=lambda r: r['ref_ml'])
spread = [by_volume[round(i * (len(by_volume) - 1) / (FIGURE_CASES - 1))]['case'] for i in range(FIGURE_CASES)]
nearest = [r['case'] for site in ('basilar', 'ICA') for r in sorted((r for r in first if r['occlusion_site'] == site), key=lambda r: r['occlusion_distance_mm'])[:FIGURE_CASES // 2 + 1]]
figures = {}
for name, cases in (('core_montage.png', spread), ('occlusion_montage.png', nearest)):
    local = os.path.join(LOCAL_DIR, name)
    if cases:
        thresholds.montage([isles24.load_case(c, slim_root, names=NAMES) for c in cases], local)
        figures[name] = local
print('Figures:', ', '.join(figures) or 'none')

## 7. Save and check
Refuses to save if a held-out ID reached any output.

In [ ]:
import shutil
import pandas as pd

per_case = pd.DataFrame([r for name in VARIANTS for r in rows[name]])
texts = {'config.json': json.dumps(config, indent=1), 'metrics.json': json.dumps(metrics, indent=1), 'per_case.csv': per_case.to_csv(index=False)}
leaked = sum(case in text for case in folds['heldout'] for text in texts.values())
if leaked:
    raise ValueError(f'{leaked} held-out IDs reached the outputs. Nothing was saved. Stop and check.')
os.makedirs(RUN_DIR, exist_ok=True)
for name in ('config.json', 'per_case.csv', 'metrics.json'):  # metrics.json last: it marks a finished run
    with open(os.path.join(RUN_DIR, name), 'w') as f:
        f.write(texts[name])
for name, local in figures.items():
    shutil.copy(local, os.path.join(RUN_DIR, name))
drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)
for name, text in texts.items():
    if open(os.path.join(RUN_DIR, name)).read() != text:
        raise ValueError(f'{name} read back from Drive differs from what was saved. Run this section again.')
drive.flush_and_unmount()
s = metrics['variants']['within_tmax6']['mean_std_over_folds']
print(f"OK: {EXPERIMENT} at {COMMIT[:7]}; {len(development)} development cases; Dice {s['dice']['mean']:.3f} ± {s['dice']['std']:.3f}; "
      f"abs core error {s['abs_error_ml']['mean']:.1f} mL; occlusion reasons {metrics['occlusion']['reasons']}; results read back.")